<div dir="rtl" align="right">

# مواضعُ الأقطابِ على خريطةِ فروةِ الرأسِ

**مجموعةُ البياناتِ**: BNCI2014-001 (تَخيّلٌ حركيٌّ)
**المُشاركُ**: 1
**القنواتُ**: 22 EEG
**معدّلُ أخذِ العيناتِ**: 250 Hz

---

## نظرةٌ عامّةٌ

هذا الدفترُ يحمّلُ مجموعةَ بياناتِ BNCI2014-001، ويَستخرجُ مواضعَ قنواتِ EEG الـ 22 من معلوماتِ MNE الخامِ، ويُصوّرُ توزيعَ الأقطابِ على خريطةِ فروةِ رأسٍ ثنائيةِ الأبعادِ بِدائرةِ رأسٍ ونقاطِ أقطابٍ مُوسومةٍ.

## ماذا يَفعلُ هذا الدفترُ

- يحمّلُ BNCI2014-001 لِلمُشاركِ 1 عبرَ MOABB
- يَستخرجُ مواضعَ القنواتِ من montage الخامِ
- يُسقطُ المواضعَ ثلاثيةَ الأبعادِ إلى ثنائيةِ الأبعادِ (منظورٌ علويٌّ)
- يَرسمُ الأقطابَ كَنقاطٍ مُوسومةٍ داخلَ دائرةِ الرأسِ

## المُخرجاتُ المُتوقّعةُ

- مخططُ رأسٍ دائريٌّ بِعلامةِ أنفٍ في الأعلى
- 22 نقطةَ قطبٍ مُوسومةٍ (Fz, FC3, C3, Cz, C4, Pz، إلخ.)
- توزيعُ 10-20 المُوسّعُ يُغطّي مناطقَ الجبهيةِ والمركزيةِ والجداريةِ
- أقطابُ القشرةِ الحركيةِ (C3, C1, Cz, C2, C4) في المركزِ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| dataset | BNCI2014_001 | مجموعةُ بياناتِ تَخيّلٍ حركيٍّ من MOABB |
| subjects | [1] | المُشاركُ 1 فقط |
| n_electrodes | 22 | أقطابُ EEG ذاتُ مواضعَ |
| projection | top | إسقاطٌ ثنائيُّ الأبعادِ من الأعلى |


</div>


<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>


In [ ]:
!pip install moabb mne scipy numpy plotly


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

MOABB تُنزّلُ البياناتِ تلقائياً عندَ أولِ استخدامٍ (~44 ميجابايت لِلمُشاركِ 1). التشغيلاتُ اللاحقةُ تَستخدمُ البياناتِ المُخزّنةَ.


</div>


In [ ]:
from moabb.datasets import BNCI2014_001
ds = BNCI2014_001()
sessions = ds.get_data(subjects=[1])
subject_key = list(sessions.keys())[0]
session_dict = sessions[subject_key]
n_sessions = len(session_dict)
n_runs = len(next(iter(session_dict.values())))
first_run = next(iter(next(iter(session_dict.values())).values()))
n_channels_raw = len(first_run.ch_names)
sfreq = first_run.info['sfreq']
print(f'Subject 1: {n_sessions} sessions, {n_runs} runs/session')
print(f'Raw channels: {n_channels_raw}, Sampling rate: {sfreq} Hz')
print(f'Channel names: {first_run.ch_names}')



In [ ]:
from moabb.paradigms import MotorImagery
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=ds, subjects=[1])
print(f'X shape: {X.shape}  (n_trials, n_channels, n_samples)')
print(f'Labels shape: {labels.shape}')
print(f'Meta shape: {meta.shape}')



<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

نَطبعُ أسماءَ القنواتِ ومعدّلَ أخذِ العيناتِ.

</div>


In [ ]:
import numpy as np
unique_labels, counts = np.unique(labels, return_counts=True)
print(f'Epoch channels: {X.shape[1]}')
print(f'Epoch samples: {X.shape[2]}')
print(f'Epoch duration: {X.shape[2] / sfreq:.2f} s')
print(f'Unique labels: {list(unique_labels)}')
print(f'Trials per class: {dict(zip(unique_labels, counts))}')
print(f'Total trials: {X.shape[0]}')



<div dir="rtl" align="right">

## 4. تطبيقُ التحليلِ

نَستخرجُ مواضعَ القنواتِ ثلاثيةَ الأبعادِ ونُسقطُها إلى ثنائيةِ الأبعادِ لِخريطةِ الرأسِ.


</div>


In [ ]:
import numpy as np
montage = first_run.get_montage()
ch_pos = montage.get_positions()['ch_pos']
ch_names = [ch for ch in first_run.ch_names if ch in ch_pos]
pos = np.array([ch_pos[ch] for ch in ch_names])
pos_2d = pos[:, :2]
scale = 1.0 / np.max(np.abs(pos_2d))
pos_2d = pos_2d * scale * 0.95
print(f'Electrodes with positions: {len(ch_names)}')
print(f'Channel names: {ch_names}')



<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- الرأسُ مُمثّلٌ كَدائرةٍ بِعلامةِ أنفٍ في الأعلى
- 22 قطباً مُوسومةٌ ومُوزّعةٌ وفقاً لِنظامِ 10-20
- الأقطابُ المركزيةُ (C3, C1, Cz, C2, C4) أساسيةٌ لِلتَخيّلِ الحركيِّ
- الأقطابُ الجبهيةُ (Fz) والجداريةُ (Pz) تُؤطّرُ التوزيعَ



</div>


In [ ]:
import plotly.graph_objects as go
theta = np.linspace(0, 2*np.pi, 100)
head_x = np.cos(theta)
head_y = np.sin(theta)
fig = go.Figure()
fig.add_trace(go.Scatter(x=head_x, y=head_y, mode='lines', line=dict(color='black', width=2),
                         showlegend=False))
fig.add_trace(go.Scatter(x=[0, -0.06, 0.06], y=[1.0, 1.1, 1.1], fill='toself',
                         mode='lines', line=dict(color='black', width=1), showlegend=False))
fig.add_trace(go.Scatter(x=pos_2d[:, 0], y=pos_2d[:, 1], mode='markers+text',
                         text=ch_names, textposition='middle center',
                         marker=dict(size=22, color='#1f77b4', line=dict(color='black', width=1)),
                         textfont=dict(size=8, color='white'), showlegend=False))
fig.update_layout(title='Electrode Positions - BNCI2014-001 (22 channels)',
                  xaxis=dict(scaleanchor='y', scaleratio=1, range=[-1.2, 1.2], showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(range=[-1.2, 1.2], showgrid=False, zeroline=False, showticklabels=False),
                  width=700, height=700, plot_bgcolor='white')
fig.show()



<div dir="rtl" align="right">

## خلاصةٌ

- مجموعاتُ بياناتِ MOABB تَتضمّنُ مواضعَ أقطابٍ قياسيةً ثلاثيةَ الأبعادِ
- توزيعُ 22 قناةً يَتّبعُ نظامَ 10-20 المُوسّعَ
- مواضعُ الأقطابِ ضروريةٌ لِلتحليلِ المكانيِّ والرسمِ الطوبوغرافيِّ
- BCI لِلتَخيّلِ الحركيِّ يَعتمدُ على الأقطابِ المركزيةِ (C3, Cz, C4)



</div>
